# NER gia phả — bản cuối (22 cây train / 3 cây dev) + xuất ONNX
**Tốn tiền (GPU Colab).** Cần trong `MyDrive/family_tree_ner/`: `train_ner.py`, `export_onnx.py`, `vi_final.zip`. Kết quả: `out_final/` (trọng số) và `onnx_final.zip`.

In [ ]:
MODEL = "vinai/phobert-base"
EPOCHS, LR, BATCH = 18, 5e-5, 16
BASE = "/content/drive/MyDrive/family_tree_ner"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "transformers>=4.40" onnx onnxruntime
import torch; assert torch.cuda.is_available(), "Chưa bật GPU"
!cd /content && unzip -qo {BASE}/vi_final.zip && ls vi_final

In [ ]:
!python {BASE}/train_ner.py --model {MODEL} --data-dir /content/vi_final --output-dir /content/out_final \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH}
!mkdir -p {BASE}/out_final && cp -r /content/out_final/* {BASE}/out_final/

In [ ]:
# Xuất ONNX + int8, đối chiếu với bản gốc (trên CPU của Colab; KHÔNG phải VPS)
!python {BASE}/export_onnx.py --model-dir /content/out_final --output-dir /content/onnx_final --sample-conll /content/vi_final/dev.conll
!cd /content && zip -qr {BASE}/onnx_final.zip onnx_final && ls -la {BASE}/onnx_final.zip

In [ ]:
from google.colab import runtime
runtime.unassign()   # NGẮT RUNTIME để dừng tính tiền